In [1]:
# # First, downgrade opencv to be compatible with numpy 1.x
# !pip install 'opencv-python-headless<4.10' 'numpy>=1.23.5,<2.0'

# # Then install cellpose without GUI (since GUI dependencies force numpy 2.x)
# !pip install cellpose

# # Verify installation
# !pip list | grep -E "numpy|cellpose|opencv"

In [2]:
# !pip install 'zarr<3'
# !pip install timm


In [3]:
# ALWAYS RUN THIS FIRST!
import os
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Specify GPU 0 (out of 4 available GPUs)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

NOTEBOOK_DIR = Path("/rsrch9/home/plm/idso_fa1_pathology/codes/yshokrollahi/vitamin-p-latest")
os.chdir(NOTEBOOK_DIR)
sys.path.insert(0, str(NOTEBOOK_DIR))
print(f"✅ Working directory: {os.getcwd()}")
print(f"✅ Using GPU: {os.environ.get('CUDA_VISIBLE_DEVICES', 'Not set')}")

✅ Working directory: /rsrch9/home/plm/idso_fa1_pathology/codes/yshokrollahi/vitamin-p-latest
✅ Using GPU: 0


In [4]:
import numpy as np
import cv2
import json
import os
import torch
from pathlib import Path
import matplotlib.pyplot as plt
from cellpose import models

# ============================================================================
# 1. SETUP & MODEL LOADING
# ============================================================================
BASE_INPUT_DIR = '/rsrch9/home/plm/idso_fa1_pathology/TIER2/yasin-vitaminp/Pathology_val_dsp_bk'
GEOJSON_ROOT   = 'predictions_batch_results'
PLOT_OUTPUT_DIR = 'dice_comparison_cpsam'
os.makedirs(PLOT_OUTPUT_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Loading Cellpose-SAM (cpsam) on {device}...")

# Load Cellpose-SAM model
model_cpsam = models.CellposeModel(gpu=torch.cuda.is_available(), 
                                   pretrained_model='cpsam', 
                                   device=device)

# ============================================================================
# 2. HELPERS
# ============================================================================
def normalize_8bit(ch):
    """Normalize channel to 0-255 uint8 for Cellpose input."""
    if ch.max() > ch.min():
        ch = (ch - ch.min()) / (ch.max() - ch.min())
    return (ch * 255).astype(np.uint8)

def calculate_dice(mask1, mask2):
    intersection = np.sum((mask1 > 0) & (mask2 > 0))
    sum_masks = np.sum(mask1 > 0) + np.sum(mask2 > 0)
    return (2. * intersection) / sum_masks if sum_masks > 0 else 1.0

# ============================================================================
# 3. BATCH PROCESSING LOOP
# ============================================================================
root_path = Path(BASE_INPUT_DIR)
case_dirs = sorted([d for d in root_path.iterdir() if d.is_dir()])

all_dice_scores = []

print(f"\n🚀 Evaluating {len(case_dirs)} cases with Cellpose-SAM...\n")

for case_path in case_dirs:
    case_id = case_path.name
    
    # Paths
    mif_path = next(case_path.glob("*_mIF_crop.png"), None)
    he_path  = next(case_path.glob("*_HE_crop.png"), None)
    geojson_path = os.path.join(GEOJSON_ROOT, case_id, "HE", f"{case_id}_HE_cell.json")
    
    if not (mif_path and os.path.exists(geojson_path)):
        continue

    try:
        # --- A. Prepare Cellpose-SAM Input (mIF) ---
        mif_raw = cv2.imread(str(mif_path), cv2.IMREAD_UNCHANGED)
        H, W = mif_raw.shape[:2]
        
        nuc_ch = normalize_8bit(mif_raw[:, :, 0]) # DAPI
        cel_ch = normalize_8bit((mif_raw[:, :, 1].astype(float) + mif_raw[:, :, 2].astype(float)) / 2.0)
        
        # Cellpose-SAM expects [H, W, 3]. Green=Nuclei, Blue=Cell
        input_cpsam = np.zeros((H, W, 3), dtype=np.uint8)
        input_cpsam[..., 1] = nuc_ch
        input_cpsam[..., 2] = cel_ch

        # --- B. Predict Reference (mIF) ---
        # Diameter=None lets the model estimate or use SAM's default
        masks_cpsam, _, _ = model_cpsam.eval([input_cpsam], diameter=None, normalize=True, resample=True)
        mask_ref_mif = masks_cpsam[0]

        # --- C. Reconstruct Vitamin-P Mask (HE GeoJSON) ---
        mask_vit_p = np.zeros((H, W), dtype=np.uint8)
        with open(geojson_path, 'r') as f:
            data = json.load(f)
        for feature in data['features']:
            coords = np.array(feature['geometry']['coordinates'][0], dtype=np.int32)
            cv2.fillPoly(mask_vit_p, [coords], 1)

        # --- D. Stats and Viz ---
        score = calculate_dice(mask_ref_mif, mask_vit_p)
        all_dice_scores.append(score)
        
        # Save visualization
        img_he = cv2.cvtColor(cv2.imread(str(he_path)), cv2.COLOR_BGR2RGB)
        fig, ax = plt.subplots(1, 2, figsize=(16, 8))
        
        # Draw contours
        ref_view = img_he.copy()
        c_ref, _ = cv2.findContours((mask_ref_mif > 0).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(ref_view, c_ref, -1, (0, 255, 0), 2)
        
        vit_view = img_he.copy()
        c_vit, _ = cv2.findContours(mask_vit_p, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(vit_view, c_vit, -1, (255, 0, 0), 2)
        
        ax[0].imshow(ref_view); ax[0].set_title(f"Ref: Cellpose-SAM (mIF)"); ax[0].axis('off')
        ax[1].imshow(vit_view); ax[1].set_title(f"Pred: Vitamin-P (HE) | Dice: {score:.3f}"); ax[1].axis('off')
        
        plt.savefig(os.path.join(PLOT_OUTPUT_DIR, f"{case_id}_cpsam_compare.png"))
        plt.close()
        
        print(f"✅ {case_id}: Dice = {score:.4f}")

    except Exception as e:
        print(f"❌ Error {case_id}: {e}")

# ============================================================================
# 4. FINAL STATS
# ============================================================================
if all_dice_scores:
    print("\n" + "="*60)
    print(f"📊 SUMMARY STATISTICS (Cellpose-SAM Reference)")
    print("="*60)
    print(f"Number of Cases : {len(all_dice_scores)}")
    print(f"MEAN DICE SCORE : {np.mean(all_dice_scores):.4f}")
    print(f"STD DEV         : {np.std(all_dice_scores):.4f}")
    print("="*60)

Loading Cellpose-SAM (cpsam) on cuda...


100%|██████████| 1.15G/1.15G [00:06<00:00, 183MB/s] 



🚀 Evaluating 24 cases with Cellpose-SAM...

✅ MOS001: Dice = 0.8896
✅ MOS002: Dice = 0.8944
✅ MOS003: Dice = 0.8964
✅ MOS004: Dice = 0.9553
✅ MOS008: Dice = 0.7965
✅ MOS010: Dice = 0.9444
✅ MOS014: Dice = 0.9432
✅ MOS015: Dice = 0.8560
✅ MOS027: Dice = 0.9335
✅ MOS033: Dice = 0.7892
✅ MOS034: Dice = 0.8644
✅ MOS042: Dice = 0.8816
✅ MOS057: Dice = 0.8505
✅ MOS058: Dice = 0.7764
✅ MOS060: Dice = 0.8855
✅ MOS061: Dice = 0.8554
✅ MOS064: Dice = 0.8190
✅ MOS067: Dice = 0.8928
✅ MOS078: Dice = 0.9670
✅ MOS082: Dice = 0.9417
✅ MOS084: Dice = 0.8517
✅ MOS086: Dice = 0.8178
✅ MOS099: Dice = 0.8988
✅ MOS106: Dice = 0.8799

📊 SUMMARY STATISTICS (Cellpose-SAM Reference)
Number of Cases : 24
MEAN DICE SCORE : 0.8784
STD DEV         : 0.0523


## Cell pose Sam Flex

In [5]:
import numpy as np
import cv2
import json
import os
import torch
from pathlib import Path
import matplotlib.pyplot as plt
from cellpose import models

# ============================================================================
# 1. SETUP & MODEL LOADING
# ============================================================================
BASE_INPUT_DIR = '/rsrch9/home/plm/idso_fa1_pathology/TIER2/yasin-vitaminp/Pathology_val_dsp_bk'
# Path to the Vitamin-P Flex predictions you just generated
GEOJSON_ROOT   = 'prediction_flex' 
PLOT_OUTPUT_DIR = 'dice_comparison_flex_vs_cpsam'
os.makedirs(PLOT_OUTPUT_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Loading Cellpose-SAM (cpsam) on {device}...")

# Load Cellpose-SAM model
model_cpsam = models.CellposeModel(gpu=torch.cuda.is_available(), 
                                   pretrained_model='cpsam', 
                                   device=device)

# ============================================================================
# 2. HELPERS
# ============================================================================
def normalize_8bit(ch):
    """Normalize channel to 0-255 uint8 for Cellpose input."""
    if ch.max() > ch.min():
        ch = (ch - ch.min()) / (ch.max() - ch.min())
    return (ch * 255).astype(np.uint8)

def calculate_dice(mask1, mask2):
    intersection = np.sum((mask1 > 0) & (mask2 > 0))
    sum_masks = np.sum(mask1 > 0) + np.sum(mask2 > 0)
    return (2. * intersection) / sum_masks if sum_masks > 0 else 1.0

# ============================================================================
# 3. BATCH PROCESSING LOOP
# ============================================================================
root_path = Path(BASE_INPUT_DIR)
case_dirs = sorted([d for d in root_path.iterdir() if d.is_dir()])

all_dice_scores = []

print(f"\n🚀 Evaluating Flex (HE) vs Cellpose-SAM (mIF) for {len(case_dirs)} cases...\n")

for case_path in case_dirs:
    case_id = case_path.name
    
    # Paths
    mif_path = next(case_path.glob("*_mIF_crop.png"), None)
    he_path  = next(case_path.glob("*_HE_crop.png"), None)
    # Target the Flex specific GeoJSON
    geojson_path = os.path.join(GEOJSON_ROOT, case_id, "HE", f"{case_id}_Flex_HE_cell.json")
    
    if not (mif_path and os.path.exists(geojson_path)):
        # print(f"⚠️ Skipping {case_id}: Missing mIF or Flex GeoJSON.")
        continue

    try:
        # --- A. Prepare Cellpose-SAM Input (mIF) ---
        mif_raw = cv2.imread(str(mif_path), cv2.IMREAD_UNCHANGED)
        H, W = mif_raw.shape[:2]
        
        nuc_ch = normalize_8bit(mif_raw[:, :, 0]) # DAPI
        cel_ch = normalize_8bit((mif_raw[:, :, 1].astype(float) + mif_raw[:, :, 2].astype(float)) / 2.0)
        
        input_cpsam = np.zeros((H, W, 3), dtype=np.uint8)
        input_cpsam[..., 1] = nuc_ch
        input_cpsam[..., 2] = cel_ch

        # --- B. Predict Reference (mIF) ---
        masks_cpsam, _, _ = model_cpsam.eval([input_cpsam], diameter=None, normalize=True, resample=True)
        mask_ref_mif = (masks_cpsam[0] > 0).astype(np.uint8)

        # --- C. Reconstruct Vitamin-P Flex Mask (HE GeoJSON) ---
        mask_flex = np.zeros((H, W), dtype=np.uint8)
        with open(geojson_path, 'r') as f:
            data = json.load(f)
        for feature in data['features']:
            coords = np.array(feature['geometry']['coordinates'][0], dtype=np.int32)
            cv2.fillPoly(mask_flex, [coords], 1)

        # --- D. Stats and Viz ---
        score = calculate_dice(mask_ref_mif, mask_flex)
        all_dice_scores.append(score)
        
        # Save visualization
        img_he = cv2.cvtColor(cv2.imread(str(he_path)), cv2.COLOR_BGR2RGB)
        fig, ax = plt.subplots(1, 2, figsize=(16, 8))
        
        # Draw contours: Reference (Green), Prediction (Cyan)
        ref_view = img_he.copy()
        c_ref, _ = cv2.findContours(mask_ref_mif, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(ref_view, c_ref, -1, (0, 255, 0), 2)
        
        flex_view = img_he.copy()
        c_flex, _ = cv2.findContours(mask_flex, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(flex_view, c_flex, -1, (0, 255, 255), 2)
        
        ax[0].imshow(ref_view); ax[0].set_title(f"Ref: Cellpose-SAM (mIF)"); ax[0].axis('off')
        ax[1].imshow(flex_view); ax[1].set_title(f"Pred: Vit-P Flex (HE) | Dice: {score:.4f}"); ax[1].axis('off')
        
        plt.savefig(os.path.join(PLOT_OUTPUT_DIR, f"{case_id}_flex_vs_cpsam.png"))
        plt.close()
        
        print(f"✅ {case_id}: Dice = {score:.4f}")

    except Exception as e:
        print(f"❌ Error {case_id}: {e}")

# ============================================================================
# 4. FINAL STATS
# ============================================================================
if all_dice_scores:
    print("\n" + "="*60)
    print(f"📊 SUMMARY: FLEX (HE) vs Cellpose-SAM (mIF)")
    print("="*60)
    print(f"Number of Cases : {len(all_dice_scores)}")
    print(f"MEAN DICE SCORE : {np.mean(all_dice_scores):.4f}")
    print(f"STD DEV         : {np.std(all_dice_scores):.4f}")
    print(f"MAX DICE        : {np.max(all_dice_scores):.4f}")
    print(f"MIN DICE        : {np.min(all_dice_scores):.4f}")
    print("="*60)

Loading Cellpose-SAM (cpsam) on cuda...

🚀 Evaluating Flex (HE) vs Cellpose-SAM (mIF) for 24 cases...

✅ MOS001: Dice = 0.8161
✅ MOS002: Dice = 0.8826
✅ MOS003: Dice = 0.8906
✅ MOS004: Dice = 0.9206
✅ MOS008: Dice = 0.8362
✅ MOS010: Dice = 0.8441
✅ MOS014: Dice = 0.8997
✅ MOS015: Dice = 0.8673
✅ MOS027: Dice = 0.8526
✅ MOS033: Dice = 0.5138
✅ MOS034: Dice = 0.7434
✅ MOS042: Dice = 0.7975
✅ MOS057: Dice = 0.8312
✅ MOS058: Dice = 0.7616
✅ MOS060: Dice = 0.7003
✅ MOS061: Dice = 0.8572
✅ MOS064: Dice = 0.6400
✅ MOS067: Dice = 0.8388
✅ MOS078: Dice = 0.9388
✅ MOS082: Dice = 0.8768
✅ MOS084: Dice = 0.8436
✅ MOS086: Dice = 0.8031
✅ MOS099: Dice = 0.8422
✅ MOS106: Dice = 0.8441

📊 SUMMARY: FLEX (HE) vs Cellpose-SAM (mIF)
Number of Cases : 24
MEAN DICE SCORE : 0.8184
STD DEV         : 0.0915
MAX DICE        : 0.9388
MIN DICE        : 0.5138


## Cellpose SAM pathologhist

In [7]:
import numpy as np
import cv2
import json
import os
import torch
from pathlib import Path
import matplotlib.pyplot as plt
from cellpose import models

# ============================================================================
# 1. SETUP & MODEL LOADING
# ============================================================================
BASE_INPUT_DIR = '/rsrch9/home/plm/idso_fa1_pathology/TIER2/yasin-vitaminp/Pathology_val_dsp_bk'
PLOT_OUTPUT_DIR = 'dice_cpsam_vs_pathologist'
os.makedirs(PLOT_OUTPUT_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Loading Cellpose-SAM (cpsam) on {device}...")

model_cpsam = models.CellposeModel(gpu=torch.cuda.is_available(), 
                                   pretrained_model='cpsam', 
                                   device=device)

# ============================================================================
# 2. HELPERS
# ============================================================================
def normalize_8bit(ch):
    if ch.max() > ch.min():
        ch = (ch - ch.min()) / (ch.max() - ch.min())
    return (ch * 255).astype(np.uint8)

def calculate_dice(mask1, mask2):
    intersection = np.sum((mask1 > 0) & (mask2 > 0))
    sum_masks = np.sum(mask1 > 0) + np.sum(mask2 > 0)
    return (2. * intersection) / sum_masks if sum_masks > 0 else 1.0

def get_canonical_type(feature):
    """Matches the logic in your visualization script to find 'Cell' objects."""
    props = feature.get('properties', {})
    clf = props.get('classification', {})
    class_name = clf.get('name', '') if isinstance(clf, dict) else str(clf or '')
    obj_type = props.get('objectType', '')
    
    text = (class_name + ' ' + obj_type).lower()
    return 'Cell' if any(k in text for k in ['cytoplasm', 'cell']) else 'Nucleus'

# ============================================================================
# 3. BATCH PROCESSING LOOP
# ============================================================================
root_path = Path(BASE_INPUT_DIR)
case_dirs = sorted([d for d in root_path.iterdir() if d.is_dir()])

all_dice_scores = []

print(f"\n🚀 Evaluating Cellpose-SAM (mIF) vs Pathologist (Human GT) for {len(case_dirs)} cases...\n")

for case_path in case_dirs:
    case_id = case_path.name
    mif_path = next(case_path.glob("*_mIF_crop.png"), None)
    geojson_path = next(case_path.glob("*.geojson"), None)
    
    if not (mif_path and geojson_path):
        continue

    try:
        # --- A. Prepare Cellpose-SAM Input (mIF) ---
        mif_raw = cv2.imread(str(mif_path), cv2.IMREAD_UNCHANGED)
        H, W = mif_raw.shape[:2]
        
        nuc_ch = normalize_8bit(mif_raw[:, :, 0])
        cel_ch = normalize_8bit((mif_raw[:, :, 1].astype(float) + mif_raw[:, :, 2].astype(float)) / 2.0)
        
        input_cpsam = np.zeros((H, W, 3), dtype=np.uint8)
        input_cpsam[..., 1] = nuc_ch 
        input_cpsam[..., 2] = cel_ch 

        # --- B. Predict with Cellpose-SAM ---
        masks_cpsam, _, _ = model_cpsam.eval([input_cpsam], diameter=None, normalize=True, resample=True)
        mask_pred_cpsam = (masks_cpsam[0] > 0).astype(np.uint8)

        # --- C. Reconstruct GT (ONLY Cells) ---
        mask_gt = np.zeros((H, W), dtype=np.uint8)
        with open(geojson_path, 'r') as f:
            data = json.load(f)
        
        actual_cell_count = 0
        for feature in data.get('features', []):
            # FIX: Filter using the canonical type check
            if get_canonical_type(feature) != 'Cell':
                continue
                
            geom = feature.get('geometry', {})
            g_type = geom.get('type')
            coords = geom.get('coordinates', [])

            poly_list = [coords] if g_type == 'Polygon' else coords if g_type == 'MultiPolygon' else []
            
            for poly in poly_list:
                try:
                    pts = np.array(poly)
                    while pts.ndim > 2: pts = pts[0] # Robust un-nesting
                    if pts.ndim == 2 and pts.shape[1] == 2:
                        pts = pts.astype(np.int32)
                        cv2.fillPoly(mask_gt, [pts], 1)
                except:
                    continue
            actual_cell_count += 1

        # --- D. Stats and Viz ---
        score = calculate_dice(mask_gt, mask_pred_cpsam)
        all_dice_scores.append(score)
        
        bg_viz = cv2.cvtColor(nuc_ch, cv2.COLOR_GRAY2RGB)
        ov_gt = bg_viz.copy(); ov_pred = bg_viz.copy()
        
        c_gt, _ = cv2.findContours(mask_gt, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(ov_gt, c_gt, -1, (0, 255, 0), 2)
        
        c_pred, _ = cv2.findContours(mask_pred_cpsam, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(ov_pred, c_pred, -1, (255, 0, 255), 2)
        
        fig, ax = plt.subplots(1, 2, figsize=(16, 8))
        ax[0].imshow(ov_gt); ax[0].set_title(f"GT: Pathologist ({actual_cell_count} Cells)"); ax[0].axis('off')
        ax[1].imshow(ov_pred); ax[1].set_title(f"Pred: CPSAM | Dice: {score:.4f}"); ax[1].axis('off')
        
        plt.savefig(os.path.join(PLOT_OUTPUT_DIR, f"{case_id}_cpsam_vs_human.png"))
        plt.close()
        
        print(f"✅ {case_id}: Dice = {score:.4f} | GT Cells: {actual_cell_count}")

    except Exception as e:
        print(f"❌ Error {case_id}: {e}")

# ============================================================================
# 4. FINAL STATS
# ============================================================================
if all_dice_scores:
    print("\n" + "="*60)
    print(f"📊 SUMMARY: Cellpose-SAM (mIF) vs Pathologist (Human GT)")
    print("="*60)
    print(f"Number of Cases : {len(all_dice_scores)}")
    print(f"MEAN DICE SCORE : {np.mean(all_dice_scores):.4f}")
    print(f"STD DEV         : {np.std(all_dice_scores):.4f}")
    print("="*60)

Loading Cellpose-SAM (cpsam) on cuda...

🚀 Evaluating Cellpose-SAM (mIF) vs Pathologist (Human GT) for 24 cases...

✅ MOS001: Dice = 0.6001 | GT Cells: 183
✅ MOS002: Dice = 0.6068 | GT Cells: 244
✅ MOS003: Dice = 0.8028 | GT Cells: 239
✅ MOS004: Dice = 0.8273 | GT Cells: 255
✅ MOS008: Dice = 0.5762 | GT Cells: 124
✅ MOS010: Dice = 0.6224 | GT Cells: 90
✅ MOS014: Dice = 0.7468 | GT Cells: 215
✅ MOS015: Dice = 0.7507 | GT Cells: 385
✅ MOS027: Dice = 0.5846 | GT Cells: 126
✅ MOS033: Dice = 0.4582 | GT Cells: 20
✅ MOS034: Dice = 0.7657 | GT Cells: 225
✅ MOS042: Dice = 0.6228 | GT Cells: 202
✅ MOS057: Dice = 0.6359 | GT Cells: 183
✅ MOS058: Dice = 0.4003 | GT Cells: 54
✅ MOS060: Dice = 0.7398 | GT Cells: 132
✅ MOS061: Dice = 0.3652 | GT Cells: 56
✅ MOS064: Dice = 0.4466 | GT Cells: 45
✅ MOS067: Dice = 0.1960 | GT Cells: 58
✅ MOS078: Dice = 0.8260 | GT Cells: 688
✅ MOS082: Dice = 0.2870 | GT Cells: 76
✅ MOS084: Dice = 0.3184 | GT Cells: 62
✅ MOS086: Dice = 0.0405 | GT Cells: 7
✅ MOS099: Dice

## Cellpose sam Recall 

In [8]:
import numpy as np
import cv2
import json
import os
import torch
from pathlib import Path
import matplotlib.pyplot as plt
from cellpose import models

# ============================================================================
# 1. SETUP & MODEL LOADING
# ============================================================================
BASE_INPUT_DIR = '/rsrch9/home/plm/idso_fa1_pathology/TIER2/yasin-vitaminp/Pathology_val_dsp_bk'
PLOT_OUTPUT_DIR = 'recall_cpsam_vs_pathologist'
os.makedirs(PLOT_OUTPUT_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Loading Cellpose-SAM (cpsam) on {device}...")

model_cpsam = models.CellposeModel(gpu=torch.cuda.is_available(), 
                                   pretrained_model='cpsam', 
                                   device=device)

# ============================================================================
# 2. HELPERS
# ============================================================================
def normalize_8bit(ch):
    if ch.max() > ch.min():
        ch = (ch - ch.min()) / (ch.max() - ch.min())
    return (ch * 255).astype(np.uint8)

def calculate_masked_recall(mask_gt, mask_pred):
    """
    Calculates Pixel-Level Recall strictly within the GT mask.
    Ignores unannotated background regions.
    """
    intersection = np.sum((mask_gt > 0) & (mask_pred > 0))
    gt_area = np.sum(mask_gt > 0)
    
    if gt_area == 0: 
        return 1.0 
        
    return intersection / gt_area

def get_canonical_type(feature):
    """Matches the logic in your visualization script to find 'Cell' objects."""
    props = feature.get('properties', {})
    clf = props.get('classification', {})
    class_name = clf.get('name', '') if isinstance(clf, dict) else str(clf or '')
    obj_type = props.get('objectType', '')
    
    text = (class_name + ' ' + obj_type).lower()
    return 'Cell' if any(k in text for k in ['cytoplasm', 'cell']) else 'Nucleus'

# ============================================================================
# 3. BATCH PROCESSING LOOP
# ============================================================================
root_path = Path(BASE_INPUT_DIR)
case_dirs = sorted([d for d in root_path.iterdir() if d.is_dir()])

all_recall_scores = []

print(f"\n🚀 Evaluating Cellpose-SAM (mIF) vs Pathologist (Masked Recall) for {len(case_dirs)} cases...\n")

for case_path in case_dirs:
    case_id = case_path.name
    mif_path = next(case_path.glob("*_mIF_crop.png"), None)
    geojson_path = next(case_path.glob("*.geojson"), None)
    
    if not (mif_path and geojson_path):
        continue

    try:
        # --- A. Prepare Cellpose-SAM Input (mIF) ---
        mif_raw = cv2.imread(str(mif_path), cv2.IMREAD_UNCHANGED)
        H, W = mif_raw.shape[:2]
        
        nuc_ch = normalize_8bit(mif_raw[:, :, 0])
        cel_ch = normalize_8bit((mif_raw[:, :, 1].astype(float) + mif_raw[:, :, 2].astype(float)) / 2.0)
        
        input_cpsam = np.zeros((H, W, 3), dtype=np.uint8)
        input_cpsam[..., 1] = nuc_ch 
        input_cpsam[..., 2] = cel_ch 

        # --- B. Predict with Cellpose-SAM ---
        masks_cpsam, _, _ = model_cpsam.eval([input_cpsam], diameter=None, normalize=True, resample=True)
        mask_pred_cpsam = (masks_cpsam[0] > 0).astype(np.uint8)

        # --- C. Reconstruct GT (ONLY Cells) ---
        mask_gt = np.zeros((H, W), dtype=np.uint8)
        with open(geojson_path, 'r') as f:
            data = json.load(f)
        
        actual_cell_count = 0
        for feature in data.get('features', []):
            # FIX: Filter using the canonical type check
            if get_canonical_type(feature) != 'Cell':
                continue
                
            geom = feature.get('geometry', {})
            g_type = geom.get('type')
            coords = geom.get('coordinates', [])

            poly_list = [coords] if g_type == 'Polygon' else coords if g_type == 'MultiPolygon' else []
            
            for poly in poly_list:
                try:
                    pts = np.array(poly)
                    while pts.ndim > 2: pts = pts[0] # Robust un-nesting
                    if pts.ndim == 2 and pts.shape[1] == 2:
                        pts = pts.astype(np.int32)
                        cv2.fillPoly(mask_gt, [pts], 1)
                except:
                    continue
            actual_cell_count += 1

        # --- D. Stats and Viz ---
        score = calculate_masked_recall(mask_gt, mask_pred_cpsam)
        all_recall_scores.append(score)
        
        bg_viz = cv2.cvtColor(nuc_ch, cv2.COLOR_GRAY2RGB)
        ov_gt = bg_viz.copy(); ov_pred = bg_viz.copy()
        
        c_gt, _ = cv2.findContours(mask_gt, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(ov_gt, c_gt, -1, (0, 255, 0), 2)
        
        c_pred, _ = cv2.findContours(mask_pred_cpsam, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(ov_pred, c_pred, -1, (255, 0, 255), 2)
        
        fig, ax = plt.subplots(1, 2, figsize=(16, 8))
        ax[0].imshow(ov_gt); ax[0].set_title(f"GT: Pathologist ({actual_cell_count} Cells)"); ax[0].axis('off')
        ax[1].imshow(ov_pred); ax[1].set_title(f"Pred: CPSAM | Recall: {score:.4f}"); ax[1].axis('off')
        
        plt.savefig(os.path.join(PLOT_OUTPUT_DIR, f"{case_id}_cpsam_vs_human.png"))
        plt.close()
        
        print(f"✅ {case_id}: Recall = {score:.4f} | GT Cells: {actual_cell_count}")

    except Exception as e:
        print(f"❌ Error {case_id}: {e}")

# ============================================================================
# 4. FINAL STATS
# ============================================================================
if all_recall_scores:
    print("\n" + "="*60)
    print(f"📊 SUMMARY: Cellpose-SAM (mIF) vs Pathologist (Masked Recall)")
    print("="*60)
    print(f"Number of Cases : {len(all_recall_scores)}")
    print(f"MEAN RECALL SCORE : {np.mean(all_recall_scores):.4f}")
    print(f"STD DEV           : {np.std(all_recall_scores):.4f}")
    print("="*60)

Loading Cellpose-SAM (cpsam) on cuda...

🚀 Evaluating Cellpose-SAM (mIF) vs Pathologist (Masked Recall) for 24 cases...

✅ MOS001: Recall = 0.9106 | GT Cells: 183
✅ MOS002: Recall = 0.9087 | GT Cells: 244
✅ MOS003: Recall = 0.8685 | GT Cells: 239
✅ MOS004: Recall = 0.9542 | GT Cells: 255
✅ MOS008: Recall = 0.9570 | GT Cells: 124
✅ MOS010: Recall = 0.9733 | GT Cells: 90
✅ MOS014: Recall = 0.9674 | GT Cells: 215
✅ MOS015: Recall = 0.9543 | GT Cells: 385
✅ MOS027: Recall = 0.9152 | GT Cells: 126
✅ MOS033: Recall = 0.9466 | GT Cells: 20
✅ MOS034: Recall = 0.9482 | GT Cells: 225
✅ MOS042: Recall = 0.8731 | GT Cells: 202
✅ MOS057: Recall = 0.8299 | GT Cells: 183
✅ MOS058: Recall = 0.4160 | GT Cells: 54
✅ MOS060: Recall = 0.8570 | GT Cells: 132
✅ MOS061: Recall = 0.7940 | GT Cells: 56
✅ MOS064: Recall = 0.7628 | GT Cells: 45
✅ MOS067: Recall = 0.8530 | GT Cells: 58
✅ MOS078: Recall = 0.9731 | GT Cells: 688
✅ MOS082: Recall = 0.9278 | GT Cells: 76
✅ MOS084: Recall = 0.8728 | GT Cells: 62
✅ MOS